# 🚀 Paradigm Assist — 1-Click LoRA Fine-Tuning & Quantization Pipeline
### **Model: Qwen 2.5 1.5B-Instruct | Target: 100% Offline Mobile Deployment (4-bit Q4_K_M)**

This notebook trains a custom **Paradigm Assist** copilot model fine-tuned on:
- 📝 ISO 9001:2015 Corporate & HR Leave Application Drafting
- 💧 WTP & STP 7-Stage Horizontal Mermaid Flowcharts (`graph LR`)
- ⚡ Diesel Generator (DG) Cold Start & Electrical Safety SOPs
- 🛗 Passenger Lift Entrapment Evacuation & Manual Brake Drift
- ⏱️ Paradigm Dynamic Shift Rules (A, B, C, Double Duty >= 14h, 1 W/O per week)

**Training Time:** ~12–15 minutes on a free Google Colab T4 GPU.

## 📦 Step 1: Install High-Speed Training Dependencies (Unsloth)

In [ ]:
%%capture
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes datasets


## 🤖 Step 2: Load Qwen 2.5 1.5B Base Model in 4-bit

In [ ]:
from unsloth import FastLanguageModel  # type: ignore
import torch  # type: ignore

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Qwen 2.5 1.5B model ready with LoRA adapters!")


## 📂 Step 3: Upload or Ingest Paradigm Training Dataset

In [ ]:
from datasets import load_dataset  # type: ignore
try:
    from google.colab import files  # type: ignore
except ImportError:
    files = None  # type: ignore
import os

print("Please upload training_dataset.jsonl from your project data folder:")
if not os.path.exists("training_dataset.jsonl") and files is not None:
    uploaded = files.upload()

dataset = load_dataset("json", data_files="training_dataset.jsonl", split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Successfully loaded {len(dataset)} Paradigm instruction pairs!")


## 🏋️ Step 4: Run LoRA Fine-Tuning

In [ ]:
from trl import SFTTrainer  # type: ignore
from transformers import TrainingArguments  # type: ignore

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="paradigm_outputs",
    ),
)

trainer_stats = trainer.train()
print("Fine-tuning completed successfully!")


## 🧪 Step 5: Test Model Live (Leave Application & WTP Test)

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = "Draft a formal leave application for 3 days due to personal emergency starting next Monday."
messages = [
    {"role": "system", "content": "You are Paradigm Assist, the official enterprise operations copilot for Paradigm Integrated Facility Services. Provide courteous, respectful operational guidance."},
    {"role": "user", "content": test_prompt}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=400, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])


## 💾 Step 6: Export & Quantize Directly to GGUF (4-Bit Q4_K_M for Android APK)
This merges the LoRA adapter into the base weights and exports a single **~920MB** binary file ready to be copied into `android/app/src/main/assets/models/`.

In [ ]:
# Export to 4-bit GGUF (Quantization Q4_K_M)
model.save_pretrained_gguf("paradigm_assist_q4", tokenizer, quantization_method="q4_k_m")
print("Saved quantized 4-bit model to folder paradigm_assist_q4!")

# Download model for bundling into Android APK
try:
    from google.colab import files  # type: ignore
except ImportError:
    files = None  # type: ignore
import glob
gguf_files = glob.glob("paradigm_assist_q4/*.gguf")
if gguf_files and files is not None:
    print(f"Downloading {gguf_files[0]} to your local computer...")
    files.download(gguf_files[0])
